# Problem Set 10: AI Emotion Detection and Happiness

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due Wednesday 12/2 (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Run a facial (and audio) emotion recognition model on a video of your choice, then reflect on what AI can and cannot tell us about emotions — and on happiness.

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up — switch on the GPU

This problem set is different from the others: instead of analysing a dataset, you will **run an AI model** — a facial emotion recognition system — on a video, and then reflect on what it can and cannot do. It is the last problem set of the quarter, and it doubles as our discussion of happiness.

You have two options: point the algorithm at a **YouTube video**, or use your **own webcam** (slower, but more fun).

> ⚠️ **Before running anything:** this notebook needs a GPU. Check that the top right of the window says **"T4"**. If it does not, go to **Runtime → Change runtime type** and select **T4 GPU**, then continue.

Unlike the rest of the quarter, you are not writing the analysis yourself here: you are running a large pre-trained model that someone else built. You only need to run the cells in order, change a link, and comment one or two lines in or out. Come to office hours or email the TA if you hit any problems.

### Step 1: Install the packages

This takes a couple of minutes.

In [ ]:
!pip install -q pytubefix
!pip install -q moviepy
!pip install -q rmn
!pip install -q modelscope
!pip install -q funasr

### Step 2: Restart the runtime

Once the installs finish, restart the session so the new packages load: **Runtime → Restart session** (or Ctrl+M / Cmd+M). Then continue with the next cell — you do **not** need to re-run the install cell.

### Step 3: Load the models

This downloads the emotion-recognition models and defines the functions we will use. You are not expected to understand this code.

In [ ]:
import cv2
import os
from rmn import RMN
from pytubefix import YouTube
import moviepy.editor as mp
from tqdm import tqdm
from funasr import AutoModel
import sys
import contextlib

# from ipywebrtc import CameraStream, VideoRecorder
from IPython.display import display, Javascript
from google.colab.output import eval_js
import io
from base64 import b64decode
import subprocess
import shutil



@contextlib.contextmanager
def suppress_stdout():
    """Context manager to suppress output."""
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

def extract_top_emotion(result):
    """
    Extract and format the emotion with the highest score from the result.
    """
    emotions = result[0]["labels"]  # List of emotion labels (Chinese/English combined)
    scores = result[0]["scores"]    # List of corresponding scores

    # Extract only the English part and find the emotion with the highest score
    top_emotion = None
    max_score = -float("inf")
    for emotion, score in zip(emotions, scores):
        if "/" in emotion:
            english_label = emotion.split("/")[1]
            if score > max_score:
                max_score = score
                top_emotion = f"{english_label}: {score:.4f}"

    return top_emotion

def overlay_emotions_on_frame(frame, text, position=(50, 50)):
    """
    Overlays the given text on the video frame at the specified position.
    """
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    color = (0, 255, 0)  # Green color for text
    thickness = 2
    line_type = cv2.LINE_AA

    # Add text to the frame
    if text:
        cv2.putText(frame, text, position, font, font_scale, color, thickness, line_type)
    return frame

def process_audio_in_chunks(audio_path, chunk_duration=2):
    """
    Process the audio in chunks and return the top emotion for each chunk.
    The default chunk duration is 2 seconds.
    """
    audio_model = AutoModel(model="iic/emotion2vec_plus_large")
    audio_clip = mp.AudioFileClip(audio_path)
    duration = audio_clip.duration

    emotions_by_chunk = []
    start_time = 0
    with suppress_stdout():
        while start_time < duration:
            end_time = min(start_time + chunk_duration, duration)
            audio_chunk = audio_clip.subclip(start_time, end_time)
            chunk_path = f"chunk_{start_time:.2f}_{end_time:.2f}.wav"
            audio_chunk.write_audiofile(chunk_path)

            # Run emotion recognition on the chunk
            rec_result = audio_model.generate(chunk_path, output_dir="./outputs", granularity="utterance", extract_embedding=False)
            top_emotion = extract_top_emotion(rec_result)

            emotions_by_chunk.append(top_emotion)
            start_time += chunk_duration

            # Clean up the chunk file
            os.remove(chunk_path)

        return emotions_by_chunk, chunk_duration

def download_video(youtube_url, output_filename='downloaded_video.mp4'):
    """
    Downloads a YouTube video and saves it to the specified output filename.

    Parameters:
        youtube_url (str): The URL of the YouTube video to download.
        output_filename (str): The filename to save the downloaded video.

    Returns:
        str: The path to the downloaded video file.
    """
    # Define the on_progress callback for YouTube download
    def on_progress(stream, chunk, bytes_remaining):
        bytes_downloaded = stream.filesize - bytes_remaining
        pbar_download.update(bytes_downloaded - pbar_download.n)

    # Initialize YouTube object with progress callback
    yt = YouTube(youtube_url, on_progress_callback=on_progress)
    video_stream = yt.streams.get_highest_resolution()

    # Initialize progress bar for download
    pbar_download = tqdm(total=video_stream.filesize, unit='B', unit_scale=True, desc='Downloading video')
    video_path = video_stream.download(filename=output_filename)
    pbar_download.close()

    return video_path


def record_video(filename='video.mp4', duration=5):
    """
    Records webcam video in Google Colab for a specified duration and saves it to a file.

    Args:
        filename (str): The name of the output video file.
        duration (int): The duration of the video recording in seconds.

    Returns:
        str: The filename of the saved video.
    """
    js = Javascript("""
        async function recordVideo(duration) {
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            const recorder = new MediaRecorder(stream);
            let data = [];

            recorder.ondataavailable = (event) => data.push(event.data);
            recorder.start();

            await new Promise(resolve => setTimeout(resolve, duration * 1000));
            recorder.stop();

            await new Promise(resolve => recorder.onstop = resolve);
            stream.getTracks().forEach(track => track.stop());

            let blob = new Blob(data, {type: 'video/webm'});
            let reader = new FileReader();
            let base64data = await new Promise((resolve, reject) => {
                reader.onloadend = () => resolve(reader.result.split(',')[1]);
                reader.onerror = reject;
                reader.readAsDataURL(blob);
            });
            return base64data;
        }
    """)
    display(js)
    print(f"Recording video for {duration} seconds...")
    data = eval_js(f'recordVideo({duration})')
    binary = b64decode(data)
    temp_filename = 'temp_video.webm'
    with open(temp_filename, 'wb') as f:
        f.write(binary)
    print(f"Video saved to {temp_filename}")

    # Convert to desired format if necessary
    if filename.endswith('.mp4'):
        # Check if FFmpeg is installed
        if not shutil.which('ffmpeg'):
            print("Installing FFmpeg...")
            subprocess.check_call(['apt-get', 'install', '-y', 'ffmpeg'])
        # Convert using FFmpeg
        print("Converting video to mp4 format...")
        cmd = ['ffmpeg', '-y', '-i', temp_filename, filename]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(f"Video converted and saved to {filename}")
        # Remove the temporary file
        os.remove(temp_filename)
    else:
        # Save as the specified filename (likely .webm)
        shutil.move(temp_filename, filename)
        print(f"Video saved to {filename}")
    return filename

def process_video(video_path, output_filename='output.mp4', start_time=None, end_time=None, chunk_duration=2):
    """
    Processes the video by extracting audio (if available), inferring emotions, overlaying them on frames,
    and combining audio and video into the final output file.

    Parameters:
        video_path (str): The path to the video file to process.
        output_filename (str): The filename for the processed output video.
        start_time (float): The start time in seconds for processing.
        end_time (float): The end time in seconds for processing.
        chunk_duration (int): Duration of each audio chunk for emotion analysis.
    """
    # Step 1: Load the video clip
    print("Loading video...")
    video_clip = mp.VideoFileClip(video_path)

    # Check if the video has audio
    has_audio = video_clip.audio is not None

    if has_audio:
        # If the video has audio, extract it for the specified time segment
        print("Extracting audio...")
        if start_time is not None or end_time is not None:
            audio_clip = video_clip.audio.subclip(start_time, end_time)
        else:
            audio_clip = video_clip.audio

        audio_path = "audio.wav"
        audio_clip.write_audiofile(audio_path)
        audio_clip.close()

        # Step 2: Process the audio in chunks and get top emotions
        print(f"Inferring emotions from audio in {chunk_duration}-second chunks...")
        with suppress_stdout():
            emotions_by_chunk, chunk_duration = process_audio_in_chunks(audio_path, chunk_duration=chunk_duration)
    else:
        print("No audio track found in the video. Skipping audio extraction and emotion inference from audio.")
        emotions_by_chunk = []
        chunk_duration = None  # We won't need chunk_duration for audio processing

    video_clip.close()

    # Step 3: Initialize video processing
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 25  # Default FPS if unable to get from the video
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Calculate frame positions based on start and end times
    if start_time is not None:
        start_frame = int(start_time * fps)
    else:
        start_frame = 0

    if end_time is not None:
        end_frame = int(end_time * fps)
    else:
        end_frame = total_frames

    # Ensure frame indices are within the valid range
    start_frame = max(0, start_frame)
    end_frame = min(total_frames, end_frame)

    # Set the starting frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames_to_process = end_frame - start_frame

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video_path = 'processed_video.mp4'
    out = cv2.VideoWriter(out_video_path, fourcc, fps, (width, height))

    # Initialize RMN model
    m = RMN()

    # Process each frame with progress bar
    print("Processing video frames...")
    pbar_frames = tqdm(total=frames_to_process, desc='Processing frames')
    frame_number = start_frame

    while cap.isOpened() and frame_number < end_frame:
        ret, frame = cap.read()
        if not ret:
            break

        # Process the frame with RMN, suppressing any output
        with suppress_stdout():
            results = m.detect_emotion_for_single_frame(frame)
        frame = m.draw(frame, results)

        # Overlay emotions from audio if available
        if has_audio and emotions_by_chunk:
            # Determine which chunk the frame belongs to
            chunk_index = int((frame_number - start_frame) // (fps * chunk_duration))
            if chunk_index < len(emotions_by_chunk):
                top_emotion = emotions_by_chunk[chunk_index]
                frame = overlay_emotions_on_frame(frame, top_emotion)

        # Write the processed frame to the output video
        out.write(frame)

        # Update progress bar
        pbar_frames.update(1)
        frame_number += 1

    pbar_frames.close()

    # Release everything when job is finished
    cap.release()
    out.release()

    # Step 4: Combine processed video with extracted audio (if available)
    print("Combining video and audio...")
    processed_video_clip = mp.VideoFileClip(out_video_path)

    if has_audio:
        audio_clip = mp.AudioFileClip(audio_path)
        # Set audio to the processed video clip
        final_video = processed_video_clip.set_audio(audio_clip)
        audio_clip.close()
    else:
        # No audio to set
        final_video = processed_video_clip

    final_video.write_videofile(output_filename, codec='libx264', audio_codec='aac' if has_audio else None)

    # Close clips
    processed_video_clip.close()
    final_video.close()

    # Clean up temporary files
    os.remove(out_video_path)
    if has_audio:
        os.remove(audio_path)

    print("Processing complete. The output video is saved as", output_filename)


## Part 1: Run the algorithm on a video

Choose **one** of the two options below by commenting the other one out with `#`.

- **Option 1 — YouTube:** paste a link. If the video is long, set `start_time` and `end_time` (in seconds) to run on just part of it. If the video has sound, the algorithm also detects emotion from the audio and shows it in the bottom-left corner.
- **Option 2 — Webcam:** record yourself making faces. Keep it under 10 seconds — recording, converting and processing all take time. There is no audio in this option.

This takes roughly 5–10 minutes.

In [ ]:
# Option 1: Download the video from youtube
# video_path = download_video(
#     'https://www.youtube.com/watch?v=2hs-yt-Pmk0',  # Replace with your video link
#     output_filename='downloaded_video.mp4'
# )


# Option 2: Using webcam video, keep this under 10 seconds
video_path = record_video(
    filename='webcam.mp4',
    duration=5
)

# Step 2: Process the video
process_video(video_path, output_filename='final_output.mp4', start_time=0, end_time=60, chunk_duration=1)

## Part 2: Show your results

**Question 1.** Display **at least two frames** where the algorithm detected *different* emotions, and caption each one.

Run the cell below to pull frames out of your output video. Change the `seconds=` values to pick moments you want to show — run it as many times as you like.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_frame(seconds, video="final_output.mp4"):
    """Display the annotated frame at a given time (in seconds) of the output video."""
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(seconds * fps))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        print(f"Could not read a frame at {seconds}s — is the video shorter than that?")
        return
    plt.figure(figsize=(9, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"t = {seconds}s")
    plt.show()

show_frame(seconds=2)
show_frame(seconds=5)

**Captions.** Describe what each frame shows — which emotion the algorithm reported, at what confidence, and whether you agree with it. (If you used a YouTube video, credit it here.)

*Example: "Left: 'Happy' at 99% confidence — I do indeed look happy. Right: 'Disgust' at 80% — I was not feeling disgust, so the algorithm is wrong here."*

**Your captions:** *[your answer here]*

## Part 3: Free response

Evaluate the following statements. Answers should be a few sentences up to a **maximum of two paragraphs** each. Do not go overboard.

**Question 2.** The algorithm accurately determined the facial emotions shown in the video. (If there was audio: the audio emotion recognition was accurate.)

**Answer:** *[your answer here]*

**Question 3.** It will one day be possible to accurately determine a person's emotional state through their face, speech patterns, and body language.

**Answer:** *[your answer here]*

**Question 4.** Relying on AI for emotional support — chatbots, virtual companions — can be a healthy, viable alternative to human relationships.

**Answer:** *[your answer here]*

**Question 5.** Happiness is due more to internal factors such as personal growth, gratitude, mindset, and resilience than to external factors such as relationships, financial security, material possessions, and social approval.

**Answer:** *[your answer here]*

Now answer the following. Again, no more than two paragraphs each.

**Question 6.** Think of the happiest months and years of your life. What were the factors that made you happy? How are you defining happiness?

**Answer:** *[your answer here]*

**Question 7.** Do you think AI will be a net positive for your own happiness and life satisfaction? Why, and/or why not?

**Answer:** *[your answer here]*

**Question 8 (optional, for bonus points).** Do bonus points make you happy? Are you happier now, knowing that you have earned an additional point?

**Answer:** *[your answer here]*

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS10"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")